# Faithful JMID on ETH/UCY

This notebook runs the official SICNav Joint-MID implementation through the portable benchmark runner. Each fold trains on `<scene>_train.pkl`, selects a checkpoint only on `<scene>_val.pkl`, then evaluates `<scene>_test.pkl` once. JMID adds scene-level SADE/SFDE to the single-agent ADE/FDE comparison.

In [2]:
%cd /Users/tahaismail/Desktop/work/lyu_lab/sicnav-diffusion-reproduction

/Users/tahaismail/Desktop/work/lyu_lab/sicnav-diffusion-reproduction


In [3]:
from argparse import Namespace
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import torch

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'experiments' / 'eth_ucy').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / 'experiments' / 'eth_ucy' / 'data'
OUTPUT_DIR = PROJECT_ROOT / 'experiments' / 'eth_ucy' / 'outputs' / 'jmid'
RUNNER_DIR = PROJECT_ROOT / 'experiments' / 'eth_ucy'
if str(RUNNER_DIR) not in sys.path:
    sys.path.insert(0, str(RUNNER_DIR))
from run_jmid_benchmark import run_benchmark
DEVICE = 'mps' if torch.backends.mps.is_available() else ('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Project: {PROJECT_ROOT}')
print(f'Device: {DEVICE}')
assert DATA_DIR.is_dir(), f'Missing ETH/UCY data: {DATA_DIR}'
assert (PROJECT_ROOT / 'trajectory_prediction' / 'safe-interactive-crowdnav' / 'sicnav_diffusion' / 'JMID' / 'MID').is_dir()

def run_jmid(*, scenes, epochs=None, batch_size=None, skip_completed=False):
    return run_benchmark(Namespace(
        config=RUNNER_DIR / 'jmid_config.yaml',
        data_dir=DATA_DIR, output_dir=OUTPUT_DIR, scenes=tuple(scenes),
        device=DEVICE, epochs=epochs, batch_size=batch_size,
        num_samples=None, sampling=None, sampling_step=None, seed=None,
        run_name='jmid', results_csv=None, skip_completed=skip_completed,
    ))

Project: /Users/tahaismail/Desktop/work/lyu_lab/sicnav-diffusion-reproduction
Device: mps


## Import Smoke Test

The import in the setup cell verifies that the direct notebook-to-runner path is available. Training is intentionally opt-in below.

In [ ]:
print('Direct JMID runner import succeeded:', run_benchmark.__module__)

## Train One Fold

Start with one held-out scene locally. On Faster, use `run_jmid_benchmark.sbatch` instead; it writes the same CSV incrementally and streams logs in real time.

In [ ]:
RUN_ONE_FOLD = False
HELDOUT_SCENE = 'eth'
EPOCHS = 1000
BATCH_SIZE = 64

if RUN_ONE_FOLD:
    one_fold_results = run_jmid(
        scenes=(HELDOUT_SCENE,), epochs=EPOCHS, batch_size=BATCH_SIZE
    )
    pd.DataFrame(one_fold_results)

In [ ]:
RUN_ALL_FOLDS = False
if RUN_ALL_FOLDS:
    all_fold_results = run_jmid(
        scenes=('eth', 'hotel', 'univ', 'zara1', 'zara2'),
        epochs=EPOCHS, batch_size=BATCH_SIZE, skip_completed=True,
    )
    pd.DataFrame(all_fold_results)

## Compare Predictors

JMID's ADE/FDE remain per-pedestrian metrics; SADE/SFDE are its scene-level metrics and should be compared across JMID folds, not against the independent toy DDPM.

In [ ]:
toy_path = PROJECT_ROOT / 'experiments' / 'eth_ucy' / 'outputs' / 'single_agent_ddpm_all_folds.csv'
mid_path = PROJECT_ROOT / 'experiments' / 'eth_ucy' / 'outputs' / 'mid' / 'mid_all_folds.csv'
jmid_path = OUTPUT_DIR / 'jmid_all_folds.csv'

frames = []
if toy_path.exists():
    toy = pd.read_csv(toy_path)
    frames.append(toy[['heldout_scene', 'ddpm_best_of_20_test_ade_m', 'ddpm_best_of_20_test_fde_m']].rename(columns={
        'ddpm_best_of_20_test_ade_m': 'toy_ddpm_ade_m',
        'ddpm_best_of_20_test_fde_m': 'toy_ddpm_fde_m',
    }))
if mid_path.exists():
    mid = pd.read_csv(mid_path)
    frames.append(mid[['heldout_scene', 'test_ade_m', 'test_fde_m']].rename(columns={
        'test_ade_m': 'imid_ade_m', 'test_fde_m': 'imid_fde_m'
    }))
if jmid_path.exists():
    jmid = pd.read_csv(jmid_path)
    frames.append(jmid[['heldout_scene', 'test_ade_m', 'test_fde_m', 'test_sade_m', 'test_sfde_m']].rename(columns={
        'test_ade_m': 'jmid_ade_m', 'test_fde_m': 'jmid_fde_m'
    }))

if frames:
    comparison = frames[0]
    for frame in frames[1:]:
        comparison = comparison.merge(frame, on='heldout_scene', how='outer')
    display(comparison.sort_values('heldout_scene'))
else:
    print('Run at least one benchmark before plotting results.')

In [ ]:
if 'comparison' in globals() and {'jmid_ade_m', 'jmid_fde_m'}.issubset(comparison.columns):
    metric_columns = [column for column in ('toy_ddpm_ade_m', 'imid_ade_m', 'jmid_ade_m', 'toy_ddpm_fde_m', 'imid_fde_m', 'jmid_fde_m') if column in comparison]
    comparison.set_index('heldout_scene')[metric_columns].plot.bar(figsize=(13, 5))
    plt.ylabel('metres; lower is better')
    plt.title('ETH/UCY best-of-20 trajectory forecasting comparison')
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()

    comparison.set_index('heldout_scene')[['test_sade_m', 'test_sfde_m']].plot.bar(figsize=(10, 4))
    plt.ylabel('metres; lower is better')
    plt.title('JMID scene-level best-of-20 metrics')
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()